- verl 的 AgentLoopWorker 是一个 Ray actor（独立进程），每个 rollout 都会在这个 actor 进程里 instantiate 一个新的 XXAgentLoop 实例
    - AgentLoopWorker 类本身只是普通 Python class；在 AgentLoopManager 里被 `ray.remote(AgentLoopWorker)` 包装后，创建出来的 agent_loop_worker_xxx 实例才是 Ray actor。

```python
# agent_loop.py

self.agent_loop_workers_class = ray.remote(AgentLoopWorker)

# self.agent_loop_workers = []
self.agent_loop_workers_class.options(...).remote(
    self.config,
    self.llm_client,
    self.teacher_client,
    self.reward_loop_worker_handles,
)

# generate_sequences
chunkes = prompts.chunk(len(self.agent_loop_workers))
outputs = await asyncio.gather(
    *[
        worker.generate_sequences.remote(chunk)
        for worker, chunk in zip(self.agent_loop_workers, chunkes, strict=True)
    ]
)


# 每个 worker actor 内部，对 chunk 里的每个样本启动一个 coroutine
for i in range(len(batch)):
    kwargs = {k: v[i] for k, v in batch.non_tensor_batch.items() if k != "__do_sample__"}
    tasks.append(
        asyncio.create_task(
            self._run_agent_loop(sample_sampling_params, trajectory_info[i], trace=trace_this_sample, **kwargs)
        )
    )
outputs = await asyncio.gather(*tasks)

# _run_agent_loop
# 新建具体 XXAgentLoop 实例就
````